# Práctica 2. Resolución de problemas con búsqueda heurística
## Ingeniería del Conocimiento    2025/2026
### Prof. Juan A. Recio García

## Parte II: Búsqueda Heurística

In [1]:
from search import *

# EJERCICIO: 8 Puzzle con búsqueda heurística

#### Para el problema del puzle de 8  vamos a definir las siguientes funciones heurísticas:
* linear(node): cuenta el número de casillas mal colocadas respecto al estado final.
* manhattan(node): suma la distancia Manhattan desde cada casilla a la posición en la que debería estar en el estado final.
* max_heuristic(node): maximo de las dos anteriores
* sqrt_manhattan(node):  raíz cuadrada de la distancia Manhattan

In [2]:
class Ocho_Puzzle(Problem):
    """Problema a del 8-puzzle.  Los estados serán tuplas de nueve elementos,
    permutaciones de los números del 0 al 8 (el 0 es el hueco). Representan la
    disposición de las fichas en el tablero, leídas por filas de arriba a
    abajo, y dentro de cada fila, de izquierda a derecha. 
    
    Las cuatro acciones del problema las representaremos mediante las cadenas:
    "Mover hueco arriba", 
    "Mover hueco abajo", 
    "Mover hueco izquierda" y
    "Mover hueco derecha"."""""

    def __init__(self, initial, goal=(1, 2, 3, 4, 5, 6, 7, 8, 0)):
        """ Define goal state and initialize a problem """
        self.goal = goal
        Problem.__init__(self, initial, goal)

    def actions(self, estado):
        pos_hueco = estado.index(0)  # busco la posicion del 0
        fila = pos_hueco //3 #Calculo su fila para saber si es la primera o la última
        columna = pos_hueco % 3 #Calculo su columna para saber si esta en la primera o en la última
        accs = list()
        
        # Otra opción sería utilizar las posiciones directamente como hablamos con Adri en clase, pero esta era la solución a la que yo me refería.
        if fila != 0:
            accs.append("Mover hueco arriba")
        if fila != 2:
            accs.append("Mover hueco abajo")
        if columna != 0:
            accs.append("Mover hueco izquierda")
        if columna != 2:
            accs.append("Mover hueco derecha")

        return accs

    def result(self, estado, accion):
        pos_hueco = estado.index(0)
        l = list(estado)
        def swap(x):
            l[pos_hueco], l[pos_hueco+x] = l[pos_hueco+x], l[pos_hueco]
        
        if (accion == "Mover hueco arriba"):
            swap(-3)
            
        elif (accion == "Mover hueco abajo"):
            swap(3)
        
        elif (accion == "Mover hueco izquierda"):
            swap(-1)   
            
        elif (accion == "Mover hueco derecha"):
            swap(1)
        
        return tuple(l)

    # Hemos añadido las heurísticas a la clase para poder acceder al atributo goal
    def linear(self, node):
        estado = node.state
        mal_colocadas =0
        
        for i, valor in enumerate(estado):
            if valor != 0 and valor != self.goal[i]:
                mal_colocadas+=1
        
        return mal_colocadas
    
#              +---+---+---+
#              | 2 | 4 | 3 |
#              +---+---+---+
#              | 1 | 5 | 6 | --------> ACTUAL
#              +---+---+---+
#              | 7 | 8 | 0 |
#              +---+---+---+

#              +---+---+---+
#              | 1 | 4 | 3 |
#              +---+---+---+
#              | 2 | 7 | 6 | --------> GOAL
#              +---+---+---+
#              | 5 | 8 | 0 |
#              +---+---+---+

    def manhattan(self, node):
        estado = node.state
        distancia = 0
        
        for i, valor in enumerate(estado):
            if valor != 0:
                posicion_objetivo = self.goal.index(valor)
                
                # (0,1) --> (1,1) |||| (0,0) => (1,2)
                fila_actual = i // 3
                columna_actual = i %3
                
                fila_objetivo = posicion_objetivo //3
                columna_objetivo = posicion_objetivo % 3
                
                distancia += abs(fila_actual - fila_objetivo) + abs(columna_actual - columna_objetivo)
                
        return distancia

    def sqrt_manhattan(self, node):
        return (self.manhattan(node)**0.5) 

    def max_heuristic(self, node):
        return max(self.manhattan(node), self.linear(node))

    def h(self, node):
        """ Return the heuristic value for a given state."""
        return self.manhattan(node)
    
    def coste_de_aplicar_accion(self, s, a):
        return 1

#### Vamos a usar las implementaciones de AIMA de los algoritmos de  busqueda_primero_el_mejor y búsqueda_a_estrella (con las heurísticas anteriores) 




#### Hay que comparar los costes temporales usando %timeit y comentar los resultados.

In [3]:
####  Vamos a probar el 8-puzzle utilizando el siguiente **estado inicial** 

#              +---+---+---+
#              | 2 | 4 | 3 |
#              +---+---+---+
#              | 1 | 5 | 6 |
#              +---+---+---+
#              | 7 | 8 | H |
#              +---+---+---+

puzle = Ocho_Puzzle((2, 4, 3, 1, 5, 6, 7, 8, 0)) 

In [4]:
# Función para mostrar el estado del tablero de forma más visual
def show(s):
    print(" ".join(map(str,s[:3])))
    print(" ".join(map(str,s[3:6])))
    print(" ".join(map(str,s[6:])))

In [5]:
show(puzle.initial)

2 4 3
1 5 6
7 8 0


In [6]:
show(puzle.goal)

1 2 3
4 5 6
7 8 0


### Ahora probamos los algoritmos de búsqueda informada con la heurística Manhatan

In [7]:
# Búsqueda voraz con la heurística de Manhattan
greedy_best_first_graph_search(puzle, puzle.manhattan).solution()

['Mover hueco arriba',
 'Mover hueco izquierda',
 'Mover hueco arriba',
 'Mover hueco izquierda',
 'Mover hueco abajo',
 'Mover hueco derecha',
 'Mover hueco derecha',
 'Mover hueco abajo']

In [8]:
# Búsqueda A* con la heurística de Manhattan
astar_search(puzle,puzle.manhattan).solution()

['Mover hueco arriba',
 'Mover hueco izquierda',
 'Mover hueco arriba',
 'Mover hueco izquierda',
 'Mover hueco abajo',
 'Mover hueco derecha',
 'Mover hueco derecha',
 'Mover hueco abajo']

### Probamos las otras heurísticas

In [9]:
astar_search(puzle,puzle.linear).solution()

['Mover hueco arriba',
 'Mover hueco izquierda',
 'Mover hueco arriba',
 'Mover hueco izquierda',
 'Mover hueco abajo',
 'Mover hueco derecha',
 'Mover hueco derecha',
 'Mover hueco abajo']

In [10]:
astar_search(puzle,puzle.max_heuristic).solution()

['Mover hueco arriba',
 'Mover hueco izquierda',
 'Mover hueco arriba',
 'Mover hueco izquierda',
 'Mover hueco abajo',
 'Mover hueco derecha',
 'Mover hueco derecha',
 'Mover hueco abajo']

In [11]:
astar_search(puzle,puzle.sqrt_manhattan).solution()

['Mover hueco arriba',
 'Mover hueco izquierda',
 'Mover hueco arriba',
 'Mover hueco izquierda',
 'Mover hueco abajo',
 'Mover hueco derecha',
 'Mover hueco derecha',
 'Mover hueco abajo']

### ¿Has notado diferencias en los tiempos de ejecución? ¿y en los resultados?
Vamos a medirlo 

In [12]:
puzzle_1 = Ocho_Puzzle((2, 4, 3, 1, 5, 6, 7, 8, 0))
puzzle_2 = Ocho_Puzzle((1, 2, 3, 4, 5, 6, 0, 7, 8))
puzzle_3 = Ocho_Puzzle((1, 2, 3, 4, 5, 7, 8, 6, 0))

In [13]:
%%timeit
astar_search(puzzle_1, puzzle_1.linear)
astar_search(puzzle_2, puzzle_2.linear)
astar_search(puzzle_3, puzzle_3.linear)

696 μs ± 15.6 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


timeit es un comando de Jupiter que permite medir cuánto tarda en ejecutarse una celda:
696 microsegundos por ejecución +/- 15.6 microsegundos de variación.
Se han hecho 7 repeticiones y cada una ha ejecutado el código 1000 veces.

In [14]:
%%timeit
astar_search(puzzle_1, puzzle_1.manhattan)
astar_search(puzzle_2, puzzle_2.manhattan)
astar_search(puzzle_3, puzzle_3.manhattan)

394 μs ± 6.48 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [15]:
%%timeit
astar_search(puzzle_1, puzzle_1.sqrt_manhattan)
astar_search(puzzle_2, puzzle_2.sqrt_manhattan)
astar_search(puzzle_3, puzzle_3.sqrt_manhattan)

6.52 ms ± 83.8 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [16]:
%%timeit
astar_search(puzzle_1, puzzle_1.max_heuristic)
astar_search(puzzle_2, puzzle_2.max_heuristic)
astar_search(puzzle_3, puzzle_3.max_heuristic)

435 μs ± 7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


####   Escribe aquí tus conclusiones sobre qué heurística te parece mejor para este problema. Para ello, ordena las heurísticas según sean más informadas que otras. P.e.: linear > manhatan > sqrt_manhattan > max_heuristic. Razona la respuesta



linear > sqrt_manhattan > max_heuristic > manhattan 

La mejor heurística para este problema es la distancia de Manhattan porque es la que hace una búsqueda más rápida en el espacio de soluciones.